# Inferring Topics from IMDB Reviews

In [2]:
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import pandas as pd
import matplotlib.pyplot as plt

## Exploring the Dataset: [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz)

In [10]:
ROOT = r'C:\Users\VijayMishra\Documents\jupyter\NLP\nmf\neuralnet\aclImdb\train\pos'

In [12]:
reviews = []
for file in os.listdir(ROOT):
    path = os.path.join(ROOT, file)
    if os.path.isfile(path):
        with open(path, 'r',encoding='utf-8', errors='ignore') as fin:
            reviews.append(fin.read())

In [13]:
len(reviews)

12500

In [14]:
for i in range(3):
    print(reviews[i])
    print('=' * 150)

Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I'm here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn't!
Homelessness (or Houselessness as George Carlin stated) has been an issue for years but never a plan to help those on the street that were once considered human who did everything from going to

## Feature Extraction

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

vect = TfidfVectorizer(stop_words='english')
X = vect.fit_transform(reviews)

df = pd.DataFrame(X.toarray(), columns=vect.get_feature_names_out())

## NMF Decomposition

In [18]:
N_TOPICS = 15
nmf = NMF(n_components=N_TOPICS)
W = nmf.fit_transform(X)  # Document-topic matrix
H = nmf.components_       # Topic-term matrix

C:\Users\VijayMishra\Anaconda4\Lib\site-packages\sklearn\decomposition\_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [20]:
# Top 10 words per topic

import numpy as np
import pandas as pd

# Top 10 words per topic
words = np.array(vect.get_feature_names_out())

topic_words = pd.DataFrame(
    np.zeros((N_TOPICS, 10)),
    index=[f'Topic {i + 1}' for i in range(N_TOPICS)],
    columns=[f'Word {i + 1}' for i in range(10)]
).astype(str)

for i in range(N_TOPICS):
    ix = H[i].argsort()[::-1][:10]
    topic_words.iloc[i] = words[ix]

topic_words

,Word 1,Word 2,Word 3,Word 4,Word 5,Word 6,Word 7,Word 8,Word 9,Word 10
Topic 1,br,10,ll,spoilers,world,end,simply,yes,spoiler,stars
Topic 2,movie,movies,watch,recommend,saw,10,definitely,enjoyed,actors,better
Topic 3,film,films,director,cinema,festival,scenes,work,art,plot,characters
Topic 4,like,think,just,don,really,people,know,say,didn,did
Topic 5,man,character,role,performance,war,john,does,plays,played,best
Topic 6,good,pretty,bad,really,acting,action,job,liked,nice,plot
Topic 7,series,episode,episodes,season,tv,trek,characters,shows,seasons,television
Topic 8,funny,comedy,laugh,hilarious,fun,jokes,humor,eddie,funniest,comedies
Topic 9,horror,house,creepy,scary,gore,budget,films,halloween,fans,effects
Topic 10,love,life,family,young,real,father,children,old,people,mother


In [21]:
# Create a topic mapping

topic_mapping = {
    'Topic 4': 'TV',
    'Topic 7': 'War',
    'Topic 8': 'Comedy',
    'Topic 12': 'Book Adaptation',
    'Topic 13': 'Horror',
    'Topic 15': 'Martial Arts / Action'
}

In [22]:
# Recall the document-topic matrix, W

W = pd.DataFrame(W, columns=[f'Topic {i + 1}' for i in range(N_TOPICS)])
W['max_topic'] = W.apply(lambda x: topic_mapping.get(x.idxmax()), axis=1)
W[pd.notnull(W['max_topic'])].head(10)

,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7,Topic 8,Topic 9,Topic 10,Topic 11,Topic 12,Topic 13,Topic 14,Topic 15,max_topic
1,0.023791,0.000000,0.005966,0.029943,0.020533,0.000000,0.000814,0.004787,0.000000,0.016304,0.000000,0.000000,0.000000,0.001949,0.000000,TV
15,0.000000,0.050724,0.011255,0.052250,0.010721,0.000000,0.000651,0.000000,0.008313,0.000000,0.015046,0.002271,0.012998,0.000000,0.000000,TV
18,0.032650,0.001862,0.030337,0.011299,0.010616,0.016095,0.000156,0.000000,0.005847,0.000258,0.004501,0.044059,0.000000,0.000000,0.000000,Book Adaptation
27,0.016875,0.047921,0.000174,0.051523,0.019525,0.000000,0.002438,0.000000,0.000190,0.003944,0.000899,0.000000,0.008301,0.000000,0.000000,TV
43,0.000130,0.030494,0.020964,0.040175,0.003835,0.015040,0.000000,0.000000,0.000486,0.005252,0.013374,0.000000,0.022503,0.005684,0.006577,TV
51,0.028706,0.038086,0.013813,0.048826,0.015030,0.008349,0.002802,0.000000,0.000000,0.011934,0.002207,0.001928,0.024374,0.001203,0.013288,TV
66,0.034266,0.015372,0.000926,0.004812,0.003588,0.014092,0.000763,0.000839,0.000000,0.000000,0.002915,0.059418,0.000000,0.000000,0.002707,Book Adaptation
70,0.000019,0.000000,0.013503,0.000000,0.010186,0.001632,0.000000,0.000000,0.000000,0.004096,0.014805,0.001508,0.016983,0.000000,0.016425,Horror
77,0.014697,0.000000,0.035728,0.009082,0.000000,0.006791,0.082858,0.000000,0.000000,0.000000,0.004153,0.079996,0.000000,0.000000,0.000000,War
84,0.000000,0.033882,0.000555,0.000000,0.000000,0.033005,0.000000,0.000000,0.000000,0.005136,0.010143,0.000000,0.039983,0.000000,0.034743,Horror


In [ ]:
reviews[58]